# Airbnb Open Data — EDA & Data Cleaning
Rebuilt as a single, linear, run-once notebook. Every cell runs
top-to-bottom exactly once — no duplicated cells, no cleaning steps
run out of order, and the CSV export happens in the LAST cell, after
every other fix, not in the middle.


## 1. Load the data

In [30]:
import pandas as pd
import numpy as np

# Raw string (r"...") required on Windows paths with backslashes.
path = r"C:\Users\yashv\OneDrive\Desktop\PROJECTS\air_bn\Airbnb_Open_Data.csv"  # <-- update to your actual path
df = pd.read_csv(path)

print(df.shape)
df.head()


(102599, 26)


C:\Users\yashv\AppData\Local\Temp\ipykernel_26192\4214878414.py:6: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


,id,NAME,host id,host_identity_verified,host name,neighbourhood group,neighbourhood,lat,long,country,...,service fee,minimum nights,number of reviews,last review,reviews per month,review rate number,calculated host listings count,availability 365,house_rules,license
0,1001254,Clean & quiet apt home by the park,80014485718,unconfirmed,Madaline,Brooklyn,Kensington,40.64749,-73.97237,United States,...,$193,10.0,9.0,10/19/2021,0.21,4.0,6.0,286.0,Clean up and treat the home the way you'd like...,NaN
1,1002102,Skylit Midtown Castle,52335172823,verified,Jenna,Manhattan,Midtown,40.75362,-73.98377,United States,...,$28,30.0,45.0,5/21/2022,0.38,4.0,2.0,228.0,Pet friendly but please confirm with me if the...,NaN
2,1002403,THE VILLAGE OF HARLEM....NEW YORK !,78829239556,NaN,Elise,Manhattan,Harlem,40.80902,-73.94190,United States,...,$124,3.0,0.0,NaN,NaN,5.0,1.0,352.0,"I encourage you to use my kitchen, cooking and...",NaN
3,1002755,NaN,85098326012,unconfirmed,Garry,Brooklyn,Clinton Hill,40.68514,-73.95976,United States,...,$74,30.0,270.0,7/5/2019,4.64,4.0,1.0,322.0,NaN,NaN
4,1003689,Entire Apt: Spacious Studio/Loft by central park,92037596077,verified,Lyndon,Manhattan,East Harlem,40.79851,-73.94399,United States,...,$41,10.0,9.0,11/19/2018,0.10,3.0,1.0,289.0,"Please no smoking in the house, porch or on th...",NaN


## 2. Standardize column names
Source data mixes `snake_case`, `Title Case`, and spaced names.
Fixing this once avoids KeyErrors from typos/case mismatches later.

In [31]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print(df.columns.tolist())


['id', 'name', 'host_id', 'host_identity_verified', 'host_name', 'neighbourhood_group', 'neighbourhood', 'lat', 'long', 'country', 'country_code', 'instant_bookable', 'cancellation_policy', 'room_type', 'construction_year', 'price', 'service_fee', 'minimum_nights', 'number_of_reviews', 'last_review', 'reviews_per_month', 'review_rate_number', 'calculated_host_listings_count', 'availability_365', 'house_rules', 'license']


## 3. Structural check — shape, dtypes, nulls
Baseline snapshot before any cleaning. Compare every later
`isnull().sum()` back to this one.

In [32]:
print(df.shape)
print(df.info())
print(df.isnull().sum())


(102599, 26)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102599 entries, 0 to 102598
Data columns (total 26 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   id                              102599 non-null  int64  
 1   name                            102349 non-null  object 
 2   host_id                         102599 non-null  int64  
 3   host_identity_verified          102310 non-null  object 
 4   host_name                       102193 non-null  object 
 5   neighbourhood_group             102570 non-null  object 
 6   neighbourhood                   102583 non-null  object 
 7   lat                             102591 non-null  float64
 8   long                            102591 non-null  float64
 9   country                         102067 non-null  object 
 10  country_code                    102468 non-null  object 
 11  instant_bookable                102494 non-null  object 
 12  can

### Findings
- `license`: ~99.6% missing → drop the column entirely.
- `house_rules`: ~50.6% missing → too sparse for core analysis, optional
  stretch-question use only.
- `last_review` / `reviews_per_month`: ~15,800 missing each → tested below.
- All other columns sit under 1% missing.

## 4. Drop `license`

In [33]:
df = df.drop(columns=['license'])
print(df.shape)


(102599, 25)


## 5. Test the "never reviewed" hypothesis
If `last_review` is null because a listing has zero reviews,
`number_of_reviews` for those rows should cluster near 0.

In [34]:
never_reviewed_check = df[df['last_review'].isnull()]['number_of_reviews'].describe()
print(never_reviewed_check)


count    15769.000000
mean         0.159110
std          4.841932
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max        228.000000
Name: number_of_reviews, dtype: float64


**Result:** ~75% of these rows have exactly 0 reviews — confirms the
hypothesis for most rows. But the max is 228, meaning a small tail
has real reviews with missing timing metadata — a genuine data gap,
not "never reviewed." Handled precisely in step 9 below.

## 6. Clean `price` and `service_fee` (currency strings → float)
Both columns are strings like `"$966 "` — strip the symbol and comma,
convert to float. `regex=False` is explicit and required: `.str.replace('$', ...)`
treats `$` as a regex end-of-string anchor by default on older pandas
versions, silently doing nothing. Being explicit avoids version-dependent bugs.

In [35]:
df['price'] = df['price'].str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)
df['service_fee'] = df['service_fee'].str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)

print(df[['price', 'service_fee']].describe())


               price    service_fee
count  102352.000000  102326.000000
mean      625.293536     125.026924
std       331.671614      66.325739
min        50.000000      10.000000
25%       340.000000      68.000000
50%       624.000000     125.000000
75%       913.000000     183.000000
max      1200.000000     240.000000


## 7. Outlier checks — `minimum_nights` and `availability_365`
Physically impossible values (negative nights, availability over
365 days/year) won't show up in a null check — they have to be
checked explicitly, or they silently distort every mean/groupby
downstream.

In [36]:
print("minimum_nights < 0:", (df['minimum_nights'] < 0).sum())
print("minimum_nights > 365:", (df['minimum_nights'] > 365).sum())
print("availability_365 < 0:", (df['availability_365'] < 0).sum())
print("availability_365 > 365:", (df['availability_365'] > 365).sum())

high_min_nights = df[df['minimum_nights'] > 365]
print(high_min_nights[['minimum_nights']].describe())

high_avail = df[df['availability_365'] > 365]
print(high_avail[['availability_365']].describe())
print(high_avail['room_type'].value_counts())


minimum_nights < 0: 13
minimum_nights > 365: 35
availability_365 < 0: 432
availability_365 > 365: 2782
       minimum_nights
count        35.00000
mean        882.80000
std        1045.96777
min         366.00000
25%         400.00000
50%         500.00000
75%         999.00000
max        5645.00000
       availability_365
count       2782.000000
mean         396.773904
std           64.672218
min          366.000000
25%          380.000000
50%          395.000000
75%          411.000000
max         3677.000000
room_type
Entire home/apt    1555
Private room       1184
Shared room          43
Name: count, dtype: int64


### Findings
- `minimum_nights` < 0, n=13 → negligible, drop.
- `minimum_nights` > 365, n=35 → mean 882, max 5645 (15+ years as a
  "minimum stay") → clearly entry errors. Drop, not cap.
- `availability_365` < 0, n=432 → room-type split matches overall
  dataset proportions, not concentrated → scattered noise. Drop.
- `availability_365` > 365, n=2,782 (2.7% of data) → NOT one uniform
  error. 75% sit at 366–400 (mild overshoot, likely entry slips);
  a smaller tail runs up to 3677 (structurally impossible, not a
  rounding error). Host-id check ruled out a single bulk-upload
  source (no host repeats more than twice) — this is scattered
  individual error at two different severities, so it gets two
  different treatments below: cap the mild group, drop the severe
  group, rather than one rule for all 2,782 rows.

## 8. Apply the outlier fixes

In [37]:
rows_before = len(df)

# minimum_nights: drop negative and >365 (real entry errors, not capping candidates)
df = df[(df['minimum_nights'] >= 0) & (df['minimum_nights'] <= 365)]

# availability_365: cap mild overshoot (366-400), drop severe (>400), drop negative
df.loc[(df['availability_365'] >= 366) & (df['availability_365'] <= 400), 'availability_365'] = 365
df = df[(df['availability_365'] <= 400) | (df['availability_365'].isnull())]
df = df[(df['availability_365'] >= 0) | (df['availability_365'].isnull())]

print(f"Rows before: {rows_before}, after: {len(df)}, dropped: {rows_before - len(df)}")
print(df['availability_365'].describe())  # max should now be <= 400, min >= 0
print(df['minimum_nights'].describe())    # max should now be <= 365, min >= 0


Rows before: 102599, after: 100583, dropped: 2016
count    100159.000000
mean        138.217165
std         132.007710
min           0.000000
25%           3.000000
50%          94.000000
75%         263.000000
max         365.000000
Name: availability_365, dtype: float64
count    100583.000000
mean          7.867463
std          17.050674
min           1.000000
25%           2.000000
50%           3.000000
75%           5.000000
max         365.000000
Name: minimum_nights, dtype: float64


## 9. Fix `reviews_per_month` — three-way split, not a blanket fill
Verified this splits `reviews_per_month` nulls into exactly three
non-overlapping categories (confirmed to sum correctly against the
total null count before writing the fix — do this same partition
check any time a "fix some, not all" fill is involved):

1. `number_of_reviews == 0` → real zero-review listings. Fill with 0.
2. `number_of_reviews > 0` but `reviews_per_month` still null → reviews
   exist, rate is genuinely unknown. Leave as NaN — do not fabricate a rate.
3. `number_of_reviews` itself is null → can't reason about a rate at
   all. Leave both as NaN.

Filling all nulls with 0 based only on category 1 would have silently
zeroed out real review activity in categories 2 and 3 — worth stating
explicitly because it's an easy mistake to make with a single fillna(0).

In [38]:
df.loc[
    (df['number_of_reviews'] == 0) & (df['reviews_per_month'].isna()),
    'reviews_per_month'
] = 0

# Sanity check: partition the remaining nulls into 3 categories, confirm they sum correctly
remaining = df['reviews_per_month'].isnull()
mismatch = df[remaining & (df['number_of_reviews'] > 0)]
both_null = df[remaining & (df['number_of_reviews'].isnull())]
total_remaining = remaining.sum()

print("Remaining reviews_per_month nulls:", total_remaining)
print("  - has reviews, rate unknown:", len(mismatch))
print("  - number_of_reviews also null:", len(both_null))
print("  - accounted for:", len(mismatch) + len(both_null), "(should match total_remaining)")


Remaining reviews_per_month nulls: 141
  - has reviews, rate unknown: 18
  - number_of_reviews also null: 123
  - accounted for: 141 (should match total_remaining)


## 10. Fill remaining columns — each with a justified strategy, not a default

- `service_fee`: found to be **exactly** `price * 0.20` for every non-null
  row (ratio std ~0.001) — this is a formula, not a correlation, so
  derive it rather than estimate with a mean.
- `country` / `country_code`: confirmed all null rows have lat/long inside
  NYC bounds before filling — geographic confirmation, not an assumption.
- `review_rate_number`: distribution is nearly flat across 1-5, no
  dominant mode. Used median rather than mean, since a mean (e.g. 3.4)
  would create a rating value that never actually occurs in the data.
- `calculated_host_listings_count`: mean (7.9) vs median (1.0) — an
  8x gap, meaning a handful of power-hosts skew the mean. Used median
  so the fill reflects a typical host, not a distorted average.

In [39]:
df['service_fee'] = df['service_fee'].fillna(df['price'] * 0.20)
df['country'] = df['country'].fillna('United States')
df['country_code'] = df['country_code'].fillna('US')
df['review_rate_number'] = df['review_rate_number'].fillna(df['review_rate_number'].median())
df['calculated_host_listings_count'] = df['calculated_host_listings_count'].fillna(
    df['calculated_host_listings_count'].median()
)

# price/service_fee: the only rows where service_fee couldn't be derived
# are rows where price is ALSO null — genuinely unrecoverable, drop them.
df = df.dropna(subset=['price', 'service_fee'])

print(df.isnull().sum())


id                                    0
name                                233
host_id                               0
host_identity_verified              269
host_name                           398
neighbourhood_group                  27
neighbourhood                        12
lat                                   8
long                                  8
country                               0
country_code                          0
instant_bookable                     85
cancellation_policy                  65
room_type                             0
construction_year                   186
price                                 0
service_fee                           0
minimum_nights                        0
number_of_reviews                   180
last_review                       15506
reviews_per_month                   140
review_rate_number                    0
calculated_host_listings_count        0
availability_365                    424
house_rules                       51285


## 11. Remaining nulls — documented as intentional, not oversights
At this point the only columns with nulls left should be:
- `name`, `host_name` — identifiers, not filled with fake values.
- `host_identity_verified`, `instant_bookable`, `cancellation_policy` —
  small categorical gaps, safe to drop rows or leave as "Unknown".
- `neighbourhood_group`, `neighbourhood`, `lat`, `long` — tiny counts,
  drop rows.
- `construction_year` — check distribution before deciding fill.
- `last_review`, `reviews_per_month` — resolved in step 9, remaining
  nulls are the documented "rate genuinely unknown" and "review count
  unknown" categories, left as NaN on purpose.
- `house_rules` — too sparse for core analysis, excluded by design.

Run the check below and handle any small remaining categorical/geo
nulls with a simple drop, since none of them are large enough to need
a dedicated strategy.

In [40]:
print(df.isnull().sum())

# small, low-stakes nulls — safe to drop outright
df = df.dropna(subset=['neighbourhood_group', 'neighbourhood', 'lat', 'long'])

print("\nFinal shape:", df.shape)
print(df.isnull().sum())


id                                    0
name                                233
host_id                               0
host_identity_verified              269
host_name                           398
neighbourhood_group                  27
neighbourhood                        12
lat                                   8
long                                  8
country                               0
country_code                          0
instant_bookable                     85
cancellation_policy                  65
room_type                             0
construction_year                   186
price                                 0
service_fee                           0
minimum_nights                        0
number_of_reviews                   180
last_review                       15506
reviews_per_month                   140
review_rate_number                    0
calculated_host_listings_count        0
availability_365                    424
house_rules                       51285


## Handling other reamining values
name host_name etc

In [41]:
df['name'] = df['name'].fillna('Unknown')
df['host_name'] = df['host_name'].fillna('Unknown')
df['host_identity_verified'] = df['host_identity_verified'].fillna('Unknown')
df['instant_bookable'] = df['instant_bookable'].fillna('Unknown')
df['cancellation_policy'] = df['cancellation_policy'].fillna('Unknown')

df = df.dropna(subset=['neighbourhood_group', 'neighbourhood', 'lat', 'long'])

In [42]:
print(df['construction_year'].describe())

count    100114.000000
mean       2012.490990
std           5.762631
min        2003.000000
25%        2008.000000
50%        2012.000000
75%        2018.000000
max        2022.000000
Name: construction_year, dtype: float64


Using median to fill the rest

In [43]:
df['construction_year'] = df['construction_year'].fillna(
    df['construction_year'].median()
)

In [44]:
print(df['availability_365'].describe())

count    99883.000000
mean       138.200955
std        132.008181
min          0.000000
25%          3.000000
50%         94.000000
75%        263.000000
max        365.000000
Name: availability_365, dtype: float64


In [45]:
df['availability_365']=df['availability_365'].fillna(df['availability_365'].median())

In [46]:
print(df.isnull().sum())

id                                    0
name                                  0
host_id                               0
host_identity_verified                0
host_name                             0
neighbourhood_group                   0
neighbourhood                         0
lat                                   0
long                                  0
country                               0
country_code                          0
instant_bookable                      0
cancellation_policy                   0
room_type                             0
construction_year                     0
price                                 0
service_fee                           0
minimum_nights                        0
number_of_reviews                   180
last_review                       15501
reviews_per_month                   139
review_rate_number                    0
calculated_host_listings_count        0
availability_365                      0
house_rules                       51271


### Checks for consitency or inconsistency in the number_of_reviews with other 2

In [47]:
df[df['number_of_reviews'].isna()][
    ['number_of_reviews', 'reviews_per_month', 'last_review']
].head(20)

,number_of_reviews,reviews_per_month,last_review
566,NaN,0.65,7/1/2019
1066,NaN,0.64,9/30/2018
1591,NaN,0.43,9/18/2018
2141,NaN,3.68,6/21/2019
3149,NaN,3.49,7/2/2019
4840,NaN,0.26,4/5/2019
6627,NaN,1.06,3/28/2019
7104,NaN,0.09,3/5/2015
8337,NaN,0.06,10/16/2015
10224,NaN,NaN,NaN


Droping the remaining 180 rows for number_of_reviews casue they are inconsisitent and cant be filled with mean median or NAN so its better to drop them as they are onlly what 0.18% of the total dataset and wont harm the dataset

In [48]:
df = df.dropna(subset=['number_of_reviews'])

In [49]:
print(df.isnull().sum())


id                                    0
name                                  0
host_id                               0
host_identity_verified                0
host_name                             0
neighbourhood_group                   0
neighbourhood                         0
lat                                   0
long                                  0
country                               0
country_code                          0
instant_bookable                      0
cancellation_policy                   0
room_type                             0
construction_year                     0
price                                 0
service_fee                           0
minimum_nights                        0
number_of_reviews                     0
last_review                       15378
reviews_per_month                    16
review_rate_number                    0
calculated_host_listings_count        0
availability_365                      0
house_rules                       51161


Fixing date formats that were differnt like 12/12/2002 and some are 12-12-2002

In [50]:
df['last_review'] = pd.to_datetime(
    df['last_review'],
    errors='coerce',
    dayfirst=True
)

C:\Users\yashv\AppData\Local\Temp\ipykernel_26192\3815142594.py:1: UserWarning: Parsing dates in %m/%d/%Y format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['last_review'] = pd.to_datetime(


In [51]:
print(df['last_review'].dtype)

datetime64[ns]


In [52]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 100115 entries, 0 to 102598
Data columns (total 25 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   id                              100115 non-null  int64         
 1   name                            100115 non-null  object        
 2   host_id                         100115 non-null  int64         
 3   host_identity_verified          100115 non-null  object        
 4   host_name                       100115 non-null  object        
 5   neighbourhood_group             100115 non-null  object        
 6   neighbourhood                   100115 non-null  object        
 7   lat                             100115 non-null  float64       
 8   long                            100115 non-null  float64       
 9   country                         100115 non-null  object        
 10  country_code                    100115 non-null  object      

## 12. Export — LAST cell, after every cleaning step
Exporting anywhere other than the final cell risks saving a
partially-cleaned dataframe, which is exactly what happened previously
when the export ran before the `availability_365` fix.

In [54]:
output_path = r"C:\Users\yashv\Downloads\airbnb_cleaned_v2.csv"
df.to_csv(output_path, index=False)
print(f"Saved {len(df)} rows, {df.shape[1]} columns to {output_path}")


Saved 100115 rows, 25 columns to C:\Users\yashv\Downloads\airbnb_cleaned_v2.csv
